# Autonomous CFO — Liquidity Agent Evaluation (OpenShift AI Workbench)

Evaluates the deployed **Liquidity Agent** against synthetic QA cases in `demo/qa_cases.json`.

- **Deterministic checks**: `/api/forecast`, automatic credit facility `proposal`, both human simulation decisions (`approved` and `rejected` via `/api/decision`), no-shortfall `422` guard, and payment-date what-if comparison.
- **Live LLM grounding checks**: Calls `/api/advice` and verifies that the OpenShift AI served model (`kernes-qwen` or configured `CFO_MODEL_NAME`) grounds its explanation in the backend's shortfall dates, amounts, credit limit, and relevant transactions.

### Configuration (Environment Variables)
- `APP_BASE_URL` (default: `http://127.0.0.1:8080` — set to the OpenShift HTTPS Route `https://...` in the workbench)
- `CFO_MODEL_NAME` (optional override label if the server does not return `model` in `/api/advice`)
- `CFO_REQUIRE_LIVE_ADVICE` (`1` to fail if `/api/advice` returns `503`, `0` to record as a blocker)
- `CFO_INSECURE_TLS` (`1` only when using a cluster self-signed Route certificate)

> Never hardcode or print `CFO_MODEL_TOKEN` or cluster credentials in this notebook.

In [ ]:
import json
import os
import ssl
import subprocess
import time
import urllib.error
import urllib.request
import uuid
from pathlib import Path

APP_BASE_URL = os.environ.get("APP_BASE_URL", "http://127.0.0.1:8080").rstrip("/")
REQUIRE_LIVE_ADVICE = os.environ.get("CFO_REQUIRE_LIVE_ADVICE", "0") == "1"
SSL_CTX = ssl._create_unverified_context() if os.environ.get("CFO_INSECURE_TLS") == "1" else None
ENV_MODEL_NAME = os.environ.get("CFO_MODEL_NAME", "unspecified")

def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "demo" / "qa_cases.json").is_file():
            return candidate
    raise FileNotFoundError("Could not locate demo/qa_cases.json from current working directory.")

REPO_ROOT = find_repo_root()
DEMO_DIR = REPO_ROOT / "demo"

try:
    COMMIT_SHA = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
except Exception:
    COMMIT_SHA = "unknown"

def http_call(path: str, *, method: str = "GET", body: bytes | None = None, headers: dict[str, str] | None = None, timeout: int = 120):
    req = urllib.request.Request(f"{APP_BASE_URL}{path}", data=body, method=method)
    for k, v in (headers or {}).items():
        req.add_header(k, v)
    t0 = time.perf_counter()
    try:
        with urllib.request.urlopen(req, context=SSL_CTX, timeout=timeout) as resp:
            raw = resp.read().decode("utf-8")
            ms = round((time.perf_counter() - t0) * 1000)
            ct = resp.headers.get("content-type", "")
            return resp.status, (json.loads(raw) if "application/json" in ct else raw), ms
    except urllib.error.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        ms = round((time.perf_counter() - t0) * 1000)
        try:
            return exc.code, json.loads(raw), ms
        except json.JSONDecodeError:
            return exc.code, raw, ms

def post_forecast(fields: dict[str, str], filename: str, csv_bytes: bytes):
    boundary = f"----nb-{uuid.uuid4().hex}"
    parts = []
    for k, v in fields.items():
        parts.append(f"--{boundary}\r\nContent-Disposition: form-data; name=\"{k}\"\r\n\r\n{v}\r\n".encode())
    parts.append(
        f"--{boundary}\r\nContent-Disposition: form-data; name=\"file\"; filename=\"{filename}\"\r\nContent-Type: text/csv\r\n\r\n".encode()
        + csv_bytes
        + f"\r\n--{boundary}--\r\n".encode()
    )
    return http_call(
        "/api/forecast",
        method="POST",
        body=b"".join(parts),
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
    )

def check_grounding(explanation: str, expectations: dict) -> dict:
    lower = explanation.lower()
    amounts_ok = all(any(tok.lower() in lower for tok in grp) for grp in expectations["required_amount_groups"])
    date_ok = any(tok.lower() in lower for tok in expectations["any_date_tokens"])
    tx_ok = any(tok.lower() in lower for tok in expectations["any_transaction_tokens"])
    return {"amounts_ok": amounts_ok, "date_ok": date_ok, "tx_ok": tx_ok, "all_ok": amounts_ok and date_ok and tx_ok}

print(f"Target: {APP_BASE_URL} | Repo commit: {COMMIT_SHA}")

In [ ]:
cases = json.loads((DEMO_DIR / "qa_cases.json").read_text(encoding="utf-8"))["cases"]
health_status, health_body, _ = http_call("/health")
assert health_status == 200 and health_body == {"status": "ok"}, f"Health failed: {health_status} {health_body}"
openapi_status, openapi_spec, _ = http_call("/openapi.json")
assert openapi_status == 200
uses_baseline_advice = "baseline" in openapi_spec["components"]["schemas"]["AdviceRequest"]["properties"]
print(f"Loaded {len(cases)} synthetic cases | /health OK | AdviceRequest uses baseline={uses_baseline_advice}")

In [ ]:
results = []
forecast_cache = {}

for case in cases:
    csv_bytes = (DEMO_DIR / case["csv"]).read_bytes()
    fields = {
        "starting_balance_minor": str(case["starting_balance_minor"]),
        "start_date": case["start_date"],
    }
    st, body, _ = post_forecast(fields, case["csv"], csv_bytes)
    forecast_ok = (
        st == 200
        and [d["balance_minor"] for d in body["baseline"]["days"]] == case["expected_baseline_balances"]
        and body["baseline"]["first_deficit"] == case["expected_first_deficit"]
        and body["proposal"] == case["expected_proposal"]
    )
    forecast_cache[case["id"]] = body

    decision_ok = True
    if case["expected_decisions"] is None:
        for kind in ("approved", "rejected"):
            dst, _, _ = http_call(
                "/api/decision",
                method="POST",
                body=json.dumps({"decision": kind, "baseline": body["baseline"]}).encode(),
                headers={"Content-Type": "application/json"},
            )
            decision_ok = decision_ok and (dst == case["expected_decision_status"])
    else:
        for kind, exp in case["expected_decisions"].items():
            dst, dbody, _ = http_call(
                "/api/decision",
                method="POST",
                body=json.dumps({"decision": kind, "baseline": body["baseline"]}).encode(),
                headers={"Content-Type": "application/json"},
            )
            decision_ok = decision_ok and (
                dst == 200
                and dbody.get("simulated") is True
                and dbody.get("limit_minor") == exp["limit_minor"]
                and dbody.get("largest_uncovered") == exp["largest_uncovered"]
                and [d["uncovered_minor"] for d in dbody["days"]] == exp["uncovered_minor"]
            )

    scenario_ok = True
    move = case["scenario_move"]
    if move is not None:
        mst, mbody, _ = post_forecast(
            {**fields, "move_transaction_id": move["move_transaction_id"], "move_to_date": move["move_to_date"]},
            case["csv"],
            csv_bytes,
        )
        scenario_ok = (
            mst == 200
            and mbody["baseline"] == body["baseline"]
            and [d["balance_minor"] for d in mbody["scenario"]["days"]] == move["expected_scenario_balances"]
            and mbody["scenario"]["first_deficit"] == move["expected_first_deficit"]
        )

    results.append({"id": case["id"], "forecast_ok": forecast_ok, "decision_ok": decision_ok, "scenario_ok": scenario_ok})
    assert forecast_ok and decision_ok and scenario_ok, f"Deterministic check failed for {case['id']}"
    print(f"[PASS] {case['id']}: forecast={forecast_ok} decision={decision_ok} scenario={scenario_ok}")

In [ ]:
advice_results = []
for case in cases:
    exp = case["explanation_expectations"]
    if exp is None:
        continue
    body = forecast_cache[case["id"]]
    req_payload = (
        {"baseline": body["baseline"], "transactions": body["transactions"]}
        if uses_baseline_advice
        else {"first_deficit": body["baseline"]["first_deficit"], "transactions": body["transactions"]}
    )
    ast, abody, latency_ms = http_call(
        "/api/advice",
        method="POST",
        body=json.dumps(req_payload).encode(),
        headers={"Content-Type": "application/json"},
        timeout=150,
    )
    if ast == 200:
        grounding = check_grounding(abody["explanation"], exp)
        model_reported = abody.get("model", ENV_MODEL_NAME)
        server_ms = abody.get("latency_ms", latency_ms)
        advice_results.append({
            "id": case["id"],
            "status": 200,
            "model": model_reported,
            "latency_ms": server_ms,
            "grounding": grounding,
            "snippet": abody["explanation"][:140],
        })
        print(f"[ADVICE 200] {case['id']} | model={model_reported} | latency={server_ms}ms | grounding={grounding}")
    else:
        advice_results.append({"id": case["id"], "status": ast, "latency_ms": latency_ms, "detail": abody})
        print(f"[ADVICE {ast}] {case['id']} | latency={latency_ms}ms | detail={abody}")
        if REQUIRE_LIVE_ADVICE:
            raise AssertionError(f"Live advice required but returned HTTP {ast} for {case['id']}")

In [ ]:
print("\n=== EVALUATION SCORECARD (Project Card: RESULTS AND NEXT STEPS) ===")
print(f"Deployed URL: {APP_BASE_URL}")
print(f"Commit SHA:   {COMMIT_SHA}")
for r in results:
    print(f"- {r['id']}: forecast={r['forecast_ok']}, decision={r['decision_ok']}, scenario={r['scenario_ok']}")
for ar in advice_results:
    if ar["status"] == 200:
        print(f"- {ar['id']} advice: HTTP 200 | model={ar['model']} | latency={ar['latency_ms']}ms | grounded={ar['grounding']['all_ok']}")
    else:
        print(f"- {ar['id']} advice: HTTP {ar['status']} (blocker: live model not connected/available on target)")